<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_cell_to_site_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# FULLY DATA-DRIVEN CELL-WISE ML MODEL
# PREDICT CELL POWER FIRST
# AGGREGATE PREDICTED CELL POWER AFTERWARD
# NO ENGINEERING EQUATIONS
# NO MANUAL POWER PARAMETERS
# OPTIMIZED / FASTER VERSION
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD EXCEL FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"

site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"

traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"

traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)

site_power = pd.read_excel(site_power_url)

traffic_4g = pd.read_excel(traffic_4g_url)

traffic_5g = pd.read_excel(traffic_5g_url)

# ============================================================
# CONVERT DATETIME
# ============================================================

site_power['datetime'] = pd.to_datetime(
    site_power['datetime']
)

traffic_4g['datetime'] = pd.to_datetime(
    traffic_4g['datetime']
)

traffic_5g['datetime'] = pd.to_datetime(
    traffic_5g['datetime']
)

# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]

# ============================================================
# CREATE TIME FEATURES
# ============================================================

traffic_4g['hour'] = (
    traffic_4g['datetime'].dt.hour
)

traffic_5g['hour'] = (
    traffic_5g['datetime'].dt.hour
)

# ============================================================
# CREATE TECHNOLOGY COLUMN
# ============================================================

traffic_4g['technology'] = '4G'

traffic_5g['technology'] = '5G'

# ============================================================
# LTE CELL COUNTS
# ============================================================

lte_counts = (

    traffic_4g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

lte_counts.rename(
    columns={'Cell_ID': 'lte_cell_count'},
    inplace=True
)

# ============================================================
# NR CELL COUNTS
# ============================================================

nr_counts = (

    traffic_5g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

nr_counts.rename(
    columns={'Cell_ID': 'nr_cell_count'},
    inplace=True
)

# ============================================================
# MERGE COUNTS TO LTE
# ============================================================

traffic_4g = traffic_4g.merge(
    lte_counts,
    on='Site_ID',
    how='left'
)

traffic_4g = traffic_4g.merge(
    nr_counts,
    on='Site_ID',
    how='left'
)

# ============================================================
# MERGE COUNTS TO NR
# ============================================================

traffic_5g = traffic_5g.merge(
    lte_counts,
    on='Site_ID',
    how='left'
)

traffic_5g = traffic_5g.merge(
    nr_counts,
    on='Site_ID',
    how='left'
)

# ============================================================
# MERGE SITE DATABASE
# ============================================================

traffic_4g = traffic_4g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

traffic_5g = traffic_5g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

# ============================================================
# AGGREGATE TOTAL TRAFFIC
# ============================================================

lte_total = (

    traffic_4g.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

lte_total.rename(
    columns={'traffic_load_mbps': 'total_4g_traffic'},
    inplace=True
)

nr_total = (

    traffic_5g.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

nr_total.rename(
    columns={'traffic_load_mbps': 'total_5g_traffic'},
    inplace=True
)

# ============================================================
# MERGE TOTAL TRAFFIC TO LTE
# ============================================================

traffic_4g = traffic_4g.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_4g = traffic_4g.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# MERGE TOTAL TRAFFIC TO NR
# ============================================================

traffic_5g = traffic_5g.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_5g = traffic_5g.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

traffic_4g = traffic_4g.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_5g = traffic_5g.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# FILL NULLS
# ============================================================

traffic_4g.fillna(0, inplace=True)

traffic_5g.fillna(0, inplace=True)

# ============================================================
# ESTIMATE CELL TARGET POWER
# ============================================================

traffic_4g['cell_target_power'] = (

    traffic_4g['site_power']

    *

    (

        traffic_4g['traffic_load_mbps']

        /

        (

            traffic_4g['total_4g_traffic']
            +
            traffic_4g['total_5g_traffic']
            +
            1

        )

    )

)

traffic_5g['cell_target_power'] = (

    traffic_5g['site_power']

    *

    (

        traffic_5g['traffic_load_mbps']

        /

        (

            traffic_5g['total_4g_traffic']
            +
            traffic_5g['total_5g_traffic']
            +
            1

        )

    )

)

# ============================================================
# COMBINE LTE + NR DATA
# ============================================================

cell_df = pd.concat(
    [traffic_4g, traffic_5g],
    ignore_index=True
)

# ============================================================
# FEATURES
# ============================================================

features = [

    'traffic_load_mbps',

    'hour',
    'trigger_ID',

    'lte_cell_count',
    'nr_cell_count',

    'RRU_2G',
    'RRU_3G',
    'RRU_4G',

    'AAU_5G',

    'Boards_4G',
    'Boards_5G',

    'BBU5900',
    'BBU3900',
    'BBU3910'

]

X = cell_df[features]

y = cell_df['cell_target_power']

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,
    random_state=42

)

# ============================================================
# RANDOM FOREST MODEL
# ============================================================

model = RandomForestRegressor(

    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1

)

model.fit(X_train, y_train)

# ============================================================
# PREDICT CELL POWER
# ============================================================

cell_df['predicted_cell_power'] = (

    model.predict(
        cell_df[features]
    )

)

# ============================================================
# AGGREGATE PREDICTED CELL POWER
# ============================================================

final_df = (

    cell_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_cell_power']

    .sum()

)

# ============================================================
# RENAME COLUMN
# ============================================================

final_df.rename(

    columns={
        'predicted_cell_power': 'predicted_site_power'
    },

    inplace=True

)

# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

final_df = final_df.merge(

    site_power,

    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],

    how='left'

)

# ============================================================
# EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(

    final_df['site_power'],
    final_df['predicted_site_power']

)

rmse = np.sqrt(

    mean_squared_error(

        final_df['site_power'],
        final_df['predicted_site_power']

    )

)

mape = np.mean(

    np.abs(

        (

            final_df['site_power']
            -
            final_df['predicted_site_power']

        )

        /

        final_df['site_power']

    )

) * 100

r2 = r2_score(

    final_df['site_power'],
    final_df['predicted_site_power']

)

# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('CELL-WISE ML PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

# ============================================================
# ERROR CALCULATION
# ============================================================

final_df['error'] = (

    final_df['site_power']
    -
    final_df['predicted_site_power']

)

final_df['error_percentage'] = (

    np.abs(final_df['error'])
    /
    final_df['site_power']

) * 100

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    'Feature': features,
    'Importance': model.feature_importances_

})

importance_df = importance_df.sort_values(

    by='Importance',
    ascending=False

)

print('================================')
print('FEATURE IMPORTANCE')
print('================================')

print(importance_df)

# ============================================================
# EXPORT RESULTS
# ============================================================

final_df.to_excel(

    'Fully_Data_Driven_Cell_Wise_Predictions.xlsx',
    index=False

)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Fully_Data_Driven_Cell_Wise_Predictions.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================

print(final_df.head(20))

CELL-WISE ML PERFORMANCE
MAE  : 459.09
RMSE : 640.08
MAPE : 11.88 %
R2   : 0.9034
FEATURE IMPORTANCE
              Feature  Importance
0   traffic_load_mbps    0.813499
2          trigger_ID    0.058654
4       nr_cell_count    0.022292
10          Boards_5G    0.021023
11            BBU5900    0.017961
8              AAU_5G    0.013409
5              RRU_2G    0.012382
7              RRU_4G    0.011781
1                hour    0.011604
3      lte_cell_count    0.009338
6              RRU_3G    0.005784
13            BBU3910    0.002123
9           Boards_4G    0.000149
12            BBU3900    0.000000
OUTPUT FILE CREATED
Fully_Data_Driven_Cell_Wise_Predictions.xlsx
    Site_ID  trigger_ID        date            datetime  predicted_site_power  \
0       101           1  2026-03-01 2026-03-01 00:00:00           6681.584976   
1       101           1  2026-03-02 2026-03-02 00:00:00           7858.343282   
2       101           1  2026-03-03 2026-03-03 00:00:00           5662.500703   
